In [ ]:

import json
import os
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm


DATA_ROOT = Path("/kaggle/input/datasets")
PV_DIR    = DATA_ROOT / "emmarex/plantdisease/PlantVillage"
PDE_DIR   = DATA_ROOT / "sadmansakibmahi/plant-disease-expert/Image Data base/Image Data base"
PDOC_DIR  = DATA_ROOT / "abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset"

OUT_DIR   = Path("/kaggle/working/splits")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SAMPLES_PER_CLASS = 3000 
MIN_LAB_SAMPLES       = 100 
                               
SEED = 42

IMG_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}


CLASS_MAPPING = {
    # PlantVillage 
    "Pepper__bell___Bacterial_spot":               "Pepper___Bacterial_spot",
    "Pepper__bell___healthy":                      "Pepper___healthy",
    "Potato___Early_blight":                       "Potato___Early_blight",
    "Potato___Late_blight":                        "Potato___Late_blight",
    "Potato___healthy":                            "Potato___healthy",
    "Tomato_Bacterial_spot":                       "Tomato___Bacterial_spot",
    "Tomato_Early_blight":                         "Tomato___Early_blight",
    "Tomato_Late_blight":                          "Tomato___Late_blight",
    "Tomato_Leaf_Mold":                            "Tomato___Leaf_Mold",
    "Tomato_Septoria_leaf_spot":                   "Tomato___Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite": "Tomato___Spider_mites",
    "Tomato__Target_Spot":                         "Tomato___Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus":       "Tomato___Yellow_Leaf_Curl_Virus",
    "Tomato__Tomato_mosaic_virus":                 "Tomato___Tomato_mosaic_virus",
    "Tomato_healthy":                              "Tomato___healthy",
    # Plant Disease Expert 
    "Apple Apple scab":                            "Apple___Apple_scab",
    "Apple Black rot":                             "Apple___Black_rot",
    "Apple Cedar apple rust":                      "Apple___Cedar_apple_rust",
    "Apple healthy":                               "Apple___healthy",
    "Blueberry healthy":                           "Blueberry___healthy",
    "Cherry (including_sour) healthy":             "Cherry___healthy",
    "Cherry (including sour) Powdery mildew":      "Cherry___Powdery_mildew",
    "Common Rust in corn Leaf":                    "Corn___Common_rust",
    "Gray Leaf Spot in corn Leaf":                 "Corn___Gray_leaf_spot",
    "Blight in corn Leaf":                         "Corn___Northern_leaf_blight",
    "Corn (maize) healthy":                        "Corn___healthy",
    "Grape Black rot":                             "Grape___Black_rot",
    "Grape Esca Black Measles":                    "Grape___Esca_Black_Measles",
    "Grape Leaf blight Isariopsis Leaf Spot":      "Grape___Leaf_blight",
    "Grape healthy":                               "Grape___healthy",
    "Orange Haunglongbing Citrus greening":        "Orange___Haunglongbing",
    "Peach healthy":                               "Peach___healthy",
    "Pepper bell Bacterial spot":                  "Pepper___Bacterial_spot",
    "Pepper bell healthy":                         "Pepper___healthy",
    "Potato Early blight":                         "Potato___Early_blight",
    "Potato Late blight":                          "Potato___Late_blight",
    "Potato healthy":                              "Potato___healthy",
    "Raspberry healthy":                           "Raspberry___healthy",
    "Soybean healthy":                             "Soybean___healthy",
    "Strawberry Leaf scorch":                      "Strawberry___Leaf_scorch",
    "Strawberry healthy":                          "Strawberry___healthy",
    "Tomato Bacterial spot":                       "Tomato___Bacterial_spot",
    "Tomato Early blight":                         "Tomato___Early_blight",
    "Tomato Late blight":                          "Tomato___Late_blight",
    "Tomato Leaf Mold":                            "Tomato___Leaf_Mold",
    "Tomato Septoria leaf spot":                   "Tomato___Septoria_leaf_spot",
    "Tomato Spider mites Two spotted spider mite": "Tomato___Spider_mites",
    "Tomato Target Spot":                          "Tomato___Target_Spot",
    "Tomato Tomato mosaic virus":                  "Tomato___Tomato_mosaic_virus",
    "Tomato healthy":                              "Tomato___healthy",
    "Brown spot in rice leaf":                     "Rice___Brown_spot",
    "Bacterial leaf blight in rice leaf":          "Rice___Bacterial_leaf_blight",
    "Leaf smut in rice leaf":                      "Rice___Leaf_smut",
    "Tomato Yellow Leaf Curl Virus":               "Tomato___Yellow_Leaf_Curl_Virus",
    # PlantDoc 
    "Apple Scab Leaf":            "Apple___Apple_scab",
    "Apple rust leaf":            "Apple___Cedar_apple_rust",
    "Apple leaf":                 "Apple___healthy",
    "Blueberry leaf":             "Blueberry___healthy",
    "Cherry leaf":                "Cherry___healthy",
    "Corn leaf blight":           "Corn___Northern_leaf_blight",
    "Corn Gray leaf spot":        "Corn___Gray_leaf_spot",
    "Corn rust leaf":             "Corn___Common_rust",
    "Bell_pepper leaf spot":      "Pepper___Bacterial_spot",
    "Bell_pepper leaf":           "Pepper___healthy",
    "Potato leaf early blight":   "Potato___Early_blight",
    "Potato leaf late blight":    "Potato___Late_blight",
    "Raspberry leaf":             "Raspberry___healthy",
    "Soyabean leaf":              "Soybean___healthy",
    "Strawberry leaf":            "Strawberry___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato Early blight leaf":   "Tomato___Early_blight",
    "Tomato leaf late blight":    "Tomato___Late_blight",
    "Tomato mold leaf":           "Tomato___Leaf_Mold",
    "Tomato Septoria leaf spot":  "Tomato___Septoria_leaf_spot",
    "Tomato leaf mosaic virus":   "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus":   "Tomato___Yellow_Leaf_Curl_Virus",
    "Tomato leaf":                "Tomato___healthy",
    "grape leaf black rot":       "Grape___Black_rot",
    "grape leaf":                 "Grape___healthy",
    "Peach leaf":                 "Peach___healthy",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
}

def collect_images(root_dir: Path, source_name: str) -> list:
    records = []
    if not root_dir.exists():
        print(f"  [WARN] Path not found: {root_dir}")
        return records
    for cls_folder in tqdm(sorted(os.listdir(root_dir)), desc=source_name):
        cls_path = root_dir / cls_folder
        if not cls_path.is_dir():
            continue
        unified = CLASS_MAPPING.get(cls_folder)
        if unified is None:
            continue
        for img_path in cls_path.iterdir():
            if img_path.suffix in IMG_EXTS and img_path.stat().st_size > 0:
                records.append({
                    "image_path":     str(img_path.absolute()),
                    "label":          unified,
                    "source_dataset": source_name,
                })
    return records


def cap_per_class(df: pd.DataFrame, max_samples: int, seed: int) -> pd.DataFrame:
    return (
        df.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), max_samples), random_state=seed))
        .reset_index(drop=True)
    )

print("=" * 55)
print("  Collecting images")
print("=" * 55)

pv_records   = collect_images(PV_DIR,             "plantvillage")
pde_records  = collect_images(PDE_DIR,            "plant_disease_expert")
pdoc_train_r = collect_images(PDOC_DIR / "train", "plantdoc_train")
pdoc_test_r  = collect_images(PDOC_DIR / "test",  "plantdoc_test")

print(f"\nRaw counts:")
print(f"  PlantVillage        : {len(pv_records):>7,}")
print(f"  Plant Disease Expert: {len(pde_records):>7,}")
print(f"  PlantDoc train      : {len(pdoc_train_r):>7,}")
print(f"  PlantDoc test (OOD) : {len(pdoc_test_r):>7,}")

df_lab        = pd.DataFrame(pv_records + pde_records)
df_pdoc_train = pd.DataFrame(pdoc_train_r)
df_pdoc_test  = pd.DataFrame(pdoc_test_r)


lab_counts    = df_lab["label"].value_counts()
lab_classes   = set(lab_counts[lab_counts >= MIN_LAB_SAMPLES].index.tolist())
pdoc_classes  = set(df_pdoc_train["label"].tolist() + df_pdoc_test["label"].tolist())
valid_classes = lab_classes | pdoc_classes   # union: keep all reachable classes

print(f"\nClass breakdown:")
print(f"  Lab classes (>= {MIN_LAB_SAMPLES} samples) : {len(lab_classes)}")
print(f"  PlantDoc-exclusive classes        : {len(pdoc_classes - lab_classes)}")
print(f"  Total valid classes               : {len(valid_classes)}")

# Filter each split to valid_classes
df_lab        = df_lab[df_lab["label"].isin(valid_classes)].reset_index(drop=True)
df_pdoc_train = df_pdoc_train[df_pdoc_train["label"].isin(valid_classes)].reset_index(drop=True)
df_pdoc_test  = df_pdoc_test[df_pdoc_test["label"].isin(valid_classes)].reset_index(drop=True)

df_lab     = cap_per_class(df_lab, MAX_SAMPLES_PER_CLASS, SEED)
print(f"\nAfter {MAX_SAMPLES_PER_CLASS}/class cap: {len(df_lab):,} lab images")

train_lab, val_lab = train_test_split(
    df_lab,
    test_size=0.2,
    stratify=df_lab["label"],
    random_state=SEED,
)

train_final = pd.concat([train_lab, df_pdoc_train], ignore_index=True)
train_final = train_final.sample(frac=1, random_state=SEED).reset_index(drop=True)

overlap = set(train_final["image_path"]) & set(val_lab["image_path"])
assert len(overlap) == 0, f"Train/val overlap: {len(overlap)} images"

pdoc_leak = set(train_final["image_path"]) & set(df_pdoc_test["image_path"])
assert len(pdoc_leak) == 0, f"PlantDoc test leakage: {len(pdoc_leak)} images"

print(f"\nNo train/val overlap")
print(f" No PlantDoc test leakage")

# ---------------------------------------------------------------------------
# Final stats
# ---------------------------------------------------------------------------
print(f"\nFinal split sizes:")
print(f"  Train (lab + PlantDoc train) : {len(train_final):>7,}")
print(f"  Val   (lab only)             : {len(val_lab):>7,}")
print(f"  OOD   (PlantDoc test only)   : {len(df_pdoc_test):>7,}")
print(f"\nTrain source breakdown:")
print(train_final["source_dataset"].value_counts().to_string())
print(f"\nOOD class counts (PlantDoc test):")
print(df_pdoc_test["label"].value_counts().to_string())

# ---------------------------------------------------------------------------
# Save CSVs
# ---------------------------------------------------------------------------
train_final.to_csv( OUT_DIR / "train.csv",        index=False)
val_lab.to_csv(     OUT_DIR / "val.csv",           index=False)
df_pdoc_test.to_csv(OUT_DIR / "plantdoc_val.csv", index=False)


identity_mapping = {cls: cls for cls in sorted(valid_classes)}
with open("/kaggle/working/class_mapping.json", "w") as f:
    json.dump(identity_mapping, f, indent=2)

print(f"\nDone. Files saved:")
print(f"  {OUT_DIR}/train.csv")
print(f"  {OUT_DIR}/val.csv")
print(f"  {OUT_DIR}/plantdoc_val.csv")
print(f"  /kaggle/working/class_mapping.json")

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import sys
sys.argv = sys.argv[:1]
sys.path.insert(0, "/kaggle/working/plant-disease-classifier")
from main import main
main()

--- Environment Setup ---
Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB

--- Loading Class Mappings ---
[Dataset] 40 classes retained (dropped 0 with <100 samples).
Number of classes: 40

--- Initializing DataLoaders ---
[Loaders] train=76,420  val=18,527  plantdoc=236

--- Starting Sequential Training ---

  Training backbone: MOBILENET_V3_SMALL

[Phase 1] Freeze backbone — 5 epochs (head only)
[freeze] Trainable params: 23,080


  [P1|E001] loss=1.7311 acc=0.493 | val_f1=0.418 | plantdoc_f1=0.125 | gen=0.193  (596.3s)
  ✓ New best Gen-Score: 0.1925 — saved.


  [P1|E002] loss=0.9986 acc=0.649 | val_f1=0.499 | plantdoc_f1=0.141 | gen=0.220  (553.2s)
  ✓ New best Gen-Score: 0.2205 — saved.


  [P1|E003] loss=0.8588 acc=0.679 | val_f1=0.524 | plantdoc_f1=0.144 | gen=0.226  (555.3s)
  ✓ New best Gen-Score: 0.2265 — saved.


  [P1|E004] loss=0.8066 acc=0.689 | val_f1=0.548 | plantdoc_f1=0.157 | gen=0.244  (554.5s)
  ✓ New best Gen-Score: 0.2441 — saved.


...

  [P2|E023] loss=0.1421 acc=0.927 | val_f1=0.950 | plantdoc_f1=0.339 | gen=0.499  (588.7s)
  Early stopping: 2/6


  [P2|E024] loss=0.1397 acc=0.929 | val_f1=0.950 | plantdoc_f1=0.338 | gen=0.499  (596.0s)
  Early stopping: 3/6


  [P2|E025] loss=0.1331 acc=0.932 | val_f1=0.951 | plantdoc_f1=0.342 | gen=0.504  (602.4s)
  Early stopping: 4/6


  [P2|E026] loss=0.1297 acc=0.934 | val_f1=0.953 | plantdoc_f1=0.342 | gen=0.504  (624.3s)
  Early stopping: 5/6


  [P2|E027] loss=0.1292 acc=0.935 | val_f1=0.951 | plantdoc_f1=0.337 | gen=0.498  (623.0s)
  Early stopping: 6/6
  Early stopping triggered in Phase 2.

  Best Gen-Score : 0.5140
  Checkpoint     : /kaggle/working/checkpoints/mobilenet_v3_small_best.pt

[Memory] Clearing GPU cache after mobilenet_v3_small...
  Allocated: 0.02 GB

  TRAINING SUMMARY
  Backbone               Best Gen-Score   Epochs
  ----------------------------------------------
  mobilenet_v3_small             0.5140       27

--- Evaluation & Thresholding ---

[Ensemble] Loading checkpoints...
  Loaded mobilenet_v3_small from /kaggle/working/checkpoints/mobilenet_v3_small_best.pt  (epoch 21, score=0.5140)

[Ensemble] Evaluating on in-distribution validation set...
  Val  Acc=0.9786  Macro-F1=0.9518  Weighted-F1=0.9786

[Ensemble] Evaluating on PlantDoc (OOD)...
  PlantDoc  Acc=0.5339  Macro-F1=0.3520  Weighted-F1=0.5274

[Threshold Calibration — Youden's J on PlantDoc]
  Optimal threshold : 0.4807
  Youden's J        

In [ ]:
import shutil
from pathlib import Path

src = Path("/kaggle/input/datasets/randamansar/model-code/HackaThon/")
dst = Path("/kaggle/working/plant-disease-classifier")

dst.mkdir(parents=True, exist_ok=True)

shutil.copytree(src, dst, dirs_exist_ok=True)

print("Done. Files copied:")
for f in dst.rglob("*"):
    print(f.relative_to(dst))